In [6]:
# Pokazanie pre-processingu i wynik sieci dla przykładowego zdjęcia nieautentycznego
import cv2
import cv2.data
import matplotlib.pyplot as plt
import torch
from torchvision import transforms

In [7]:
import torch.nn as nn
import torch.nn.functional as F


class SpoofDetectionNet(nn.Module):
    def __init__(self):
        super(SpoofDetectionNet, self).__init__()
        # CONV => RELU => CONV => RELU => POOL
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding='same')
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding='same')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.dropout1 = nn.Dropout(0.25)

        # CONV => RELU => CONV => RELU => POOL
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding='same')
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding='same')
        self.bn4 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.dropout2 = nn.Dropout(0.25)

        # FC => RELU
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.bn5 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.5)

        # Warstwa wyjściowa
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)

        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)

        # FC => RELU
        x = self.flatten(x)
        x = F.relu(self.bn5(self.fc1(x)))
        x = self.dropout3(x)

        # Klasyfikator Softmax
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [8]:
# Wczytanie sieci
model = SpoofDetectionNet()
model.load_state_dict(torch.load('model_closeup.pth'))
model.eval()

In [9]:
# Wczytanie zdjęcia
image = cv2.imread("../../../Dane/NN raw/spoof/0.webp")

# Pokazanie oryginalnego obrazu
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [10]:
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_classifier.detectMultiScale(image, 1.3, 5)
face = faces[0]

(x, y, w, h) = face
x = max(0, x - w // 2)
y = max(0, y - h // 2)
w = min(image.shape[1], w * 2)
h = min(image.shape[0], h * 2)
face_area = image[y:y + h, x:x + w]

# Przetworzenie obrazu
face_area_processed = cv2.resize(face_area, (64, 64))

# Pokazanie przetworzonego obrazu
plt.imshow(cv2.cvtColor(face_area_processed, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [18]:
trf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

face_area_processed = trf(face_area).unsqueeze(0)

with torch.no_grad():
    output = model(face_area_processed)
    # 'result' będzie równy 0 lub 1, gdzie 1 oznacza twarz podstawioną
    result = torch.argmax(output, dim=1).item()
    if result == 1:
        print("Twarz podstawiona")
    else:
        print("Twarz autentyczna")